### Paso 1.- Importar Librerias

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, roc_auc_score)

### Paso 2.- Cargar Dataset

In [140]:
# ─── Cargar el dataset ────────────────────────────────────────────────────────
# 📌 ACCIÓN: Cambia la ruta al lugar donde tienes guardado el CSV
# Descarga el dataset desde: /data/processed/coffee_quality_clean.csv
# La ruta no modificarla. Y poner el mismo nombre todos.

df = pd.read_csv('../data/processed/coffee_quality_clean.csv')

print(f'✅ Datasets cargados correctamente')
print(f'📊 Dimensiones df: {df.shape[0]} filas x {df.shape[1]} columnas')

✅ Datasets cargados correctamente
📊 Dimensiones df: 1518 filas x 18 columnas


### Paso 3.- Preprocesamiento de Datos

#### Qué hacer con las siguientes columnas : Country of Origin, Altitude, Harvest Year, Variety, Processing Method y Color

#### Mantengo Altitude y Color y borro las demás junto a Uniformity, Clean Cup y Sweetness

In [141]:
# ─── DECISIÓN: Qué columnas eliminar ─────────────────────────────────────────
# Estas columnas son metadata (identificadores, nombres) que no aportan al modelo
# 📌 ACCIÓN: Revisa la lista de columnas y ajusta eliminando lo que consideres
df = df.drop(['Country of Origin', 'Harvest Year', 'Variety', 'Processing Method', 'Uniformity', 'Clean Cup', 'Sweetness'], axis=1)

In [142]:
# Agrupar variantes de color
mapa_color = {
    'yellow-green': 'Yellow Green',
    'yello-green':  'Yellow Green',
    'yellow- green': 'Yellow Green',
    'yellow green': 'Yellow Green',
    'green': 'Green',
    'browish-green': 'Blue-Green',
    'greenish': 'Green',
    'yellowish': 'Yellow',
    'brownish': 'Brown',
    'pale yellow': 'Yellow',
    'blue-green': 'Blue-Green',
    'bluish-green': 'Bluish-Green'
}
df['Color'] = df['Color'].replace(mapa_color)

In [143]:
df['Color'].unique()

<StringArray>
['Green', 'Blue-Green', 'Yellow', 'Yellow Green', 'Brown', 'Bluish-Green',
 nan]
Length: 7, dtype: str

In [144]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1518 entries, 0 to 1517
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Altitude              1290 non-null   str    
 1   Aroma                 1518 non-null   float64
 2   Flavor                1518 non-null   float64
 3   Aftertaste            1518 non-null   float64
 4   Acidity               1518 non-null   float64
 5   Body                  1518 non-null   float64
 6   Balance               1518 non-null   float64
 7   Category One Defects  1518 non-null   int64  
 8   Color                 1251 non-null   str    
 9   Category Two Defects  1518 non-null   int64  
 10  Specialty             1518 non-null   str    
dtypes: float64(6), int64(2), str(3)
memory usage: 130.6 KB


#### Borro columnas Altitude y Color para hacer pruebas antes de hacer label-encoding

In [ ]:
#df = df.drop(['Altitude', 'Color'], axis=1)

#### Que hacer con las columnas Altitude y Color

#### Tratamiento de Color con MODA -> one-hot encoding

In [145]:
# Estudio de nulos de Color
df['Color'].isna().sum()

np.int64(267)

In [ ]:
# Rellenar los NaN con la moda

# Calcular la moda y rellenar NaN
# moda = df['Color'].mode()[0]
# print(moda)

# df['Color'] = df['Color'].fillna(moda)

Green


In [146]:
# Usar one hot encoding
# Aplicar get_dummies
df = pd.get_dummies(df, columns=['Color'])
df.head(3)

,Altitude,Aroma,Flavor,Aftertaste,Acidity,Body,Balance,Category One Defects,Category Two Defects,Specialty,Color_Blue-Green,Color_Bluish-Green,Color_Brown,Color_Green,Color_Yellow,Color_Yellow Green
0,1700-1930,8.58,8.50,8.42,8.58,8.25,8.42,0,3,Specialty,False,False,False,True,False,False
1,1200,8.50,8.50,7.92,8.00,7.92,8.25,0,0,Specialty,True,False,False,False,False,False
2,1300,8.33,8.42,8.08,8.17,7.92,8.17,0,2,Specialty,False,False,False,False,True,False


In [147]:
df.isna().sum()

Altitude                228
Aroma                     0
Flavor                    0
Aftertaste                0
Acidity                   0
Body                      0
Balance                   0
Category One Defects      0
Category Two Defects      0
Specialty                 0
Color_Blue-Green          0
Color_Bluish-Green        0
Color_Brown               0
Color_Green               0
Color_Yellow              0
Color_Yellow Green        0
dtype: int64

In [148]:
df.shape

(1518, 16)

#### Tratamiento de Altitude

In [149]:
# Ver datos de las columnas
df['Altitude'].unique()

<StringArray>
['1700-1930',      '1200',      '1300',      '1900', '1850-2100',      '1668',
      '1250', '1400-1700', '1800-2200',      '2000',
 ...
    '4287.0',     '808.0',    '1187.5',     '250.0',    '3845.0',    '1022.0',
    '1264.0',    '3500.0',    '1280.0',    '1140.0']
Length: 299, dtype: str

In [150]:
# Rellenar los NaN con la moda

# Calcular la moda y rellenar NaN
moda = df['Altitude'].mode()[0]
print(moda)

df['Altitude'] = df['Altitude'].fillna(moda)

1200.0


In [151]:
def parse_value(x):
    if '-' in x:
        a, b = x.split('-')
        c = (float(a) + float(b)) / 2
        return str(c)
    if '~' in x:
        a, b = x.split('~')
        c = (float(a) + float(b)) / 2
        return str(c)
    if 'A' in x:
        a, b = x.split('A')
        c = (float(a) + float(b)) / 2
        return str(c)
    return x


df['Altitude'] = df['Altitude'].apply(parse_value)

In [152]:
df['Altitude'] = df['Altitude'].astype(float)

In [ ]:
#df = df.drop(['Altitude'], axis=1)

In [153]:
df.sample(8)

,Altitude,Aroma,Flavor,Aftertaste,Acidity,Body,Balance,Category One Defects,Category Two Defects,Specialty,Color_Blue-Green,Color_Bluish-Green,Color_Brown,Color_Green,Color_Yellow,Color_Yellow Green
1163,1400.0,7.08,7.33,7.42,7.33,7.33,7.50,0,3,No Specialty,False,False,False,True,False,False
675,700.0,7.75,7.50,7.42,7.75,7.58,7.58,0,5,Specialty,False,False,False,True,False,False
456,1775.0,7.67,7.67,7.50,7.50,7.67,7.67,0,3,Specialty,False,False,False,True,False,False
1431,1200.0,6.83,7.00,6.92,7.08,6.92,6.83,3,4,No Specialty,False,False,False,True,False,False
857,1150.0,7.67,7.58,7.58,7.75,7.67,6.83,0,0,Specialty,False,False,False,True,False,False
441,1200.0,8.00,7.75,7.50,7.58,7.92,7.67,0,4,Specialty,False,False,False,True,False,False
703,890.0,7.58,7.50,7.50,7.75,7.67,7.50,0,4,Specialty,False,False,False,True,False,False
798,1500.0,7.50,7.67,7.42,7.67,7.50,7.50,0,2,Specialty,False,False,False,True,False,False


#### Convertir la columna Specialty en 1 y 0

In [154]:
df['Specialty'] = df['Specialty'].map({'Specialty': 1, 'No Specialty': 0})
df.sample(4)

,Altitude,Aroma,Flavor,Aftertaste,Acidity,Body,Balance,Category One Defects,Category Two Defects,Specialty,Color_Blue-Green,Color_Bluish-Green,Color_Brown,Color_Green,Color_Yellow,Color_Yellow Green
751,1650.00,8.00,7.50,7.50,7.58,7.33,7.42,0,0,1,False,False,False,True,False,False
973,1700.00,7.75,7.42,7.42,7.58,7.33,7.42,0,1,0,False,False,False,True,False,False
872,1310.64,7.83,7.67,7.17,7.33,7.50,7.67,0,0,1,False,False,False,True,False,False
961,1200.00,7.67,7.58,7.42,7.00,7.50,7.58,0,1,0,False,False,False,True,False,False


#### Dividir Datos en Entrenamiento y Test

In [155]:
# 1. Seleccionar características (X) y variable objetivo (y)
X = df.drop('Specialty', axis=1)
y = df['Specialty']

# 2. Dividir el conjunto de datos (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Paso 4.- Entrenar el modelo

In [ ]:
# # 3. Crear y entrenar el modelo
# clf = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
# clf.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current no

### Paso 5.- Evaluar el modelo

In [ ]:
# # 4. Hacer predicciones
# y_pred = clf.predict(X_test)
# y_pred_proba = clf.predict_proba(X_test)[:, 1] # Para AUC-ROC

# # 5. Evaluar el modelo
# print("Exactitud:", accuracy_score(y_test, y_pred))
# print("\nMatriz de Confusión:\n", confusion_matrix(y_test, y_pred))
# print("\nInforme de Clasificación:\n", classification_report(y_test, y_pred))
# print("AUC-ROC:", roc_auc_score(y_test, y_pred_proba))

Exactitud: 0.9013157894736842

Matriz de Confusión:
 [[116  25]
 [  5 158]]

Informe de Clasificación:
               precision    recall  f1-score   support

No Specialty       0.96      0.82      0.89       141
   Specialty       0.86      0.97      0.91       163

    accuracy                           0.90       304
   macro avg       0.91      0.90      0.90       304
weighted avg       0.91      0.90      0.90       304

AUC-ROC: 0.9470913283731454


### Paso 6.- Visualizar el árbol

In [ ]:
# from sklearn.tree import plot_tree
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12,8))
# plot_tree(clf, filled=True, feature_names=df.columns, class_names=list(df['Specialty']))
# plt.show()

### Función para evaluar cualquier modelo

#### Preparar las librerías

In [156]:
# Para cross_val_score y StratifiedKFold
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Para precision_score, recall_score y f1_score
from sklearn.metrics import precision_score, recall_score, f1_score

In [157]:
# ─── Función para evaluar cualquier modelo ────────────────────────────────────
def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te, cv=5):
    """
    Entrena el modelo, predice en test y devuelve un diccionario con todas las métricas.
    También hace validación cruzada para ver si el modelo es estable.
    """
    # Entrenar
    modelo.fit(X_tr, y_tr)
    
    # Predecir en test
    y_pred = modelo.predict(X_te)
    y_prob = modelo.predict_proba(X_te)[:, 1]  # probabilidad de la clase positiva (Specialty)
    
    # Validación cruzada con F1 (más fiable con dataset pequeño y desbalanceado)
    cv_scores = cross_val_score(
        modelo, X_tr, y_tr,
        cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
        scoring='f1'
    )
    
    return {
        'Modelo'     : nombre,
        'Accuracy'   : accuracy_score(y_te, y_pred),
        #'Precision'  : precision_score(y_te, y_pred),
        'Precision'  : precision_score(y_te, y_pred),
        'Recall'     : recall_score(y_te, y_pred),
        'F1'         : f1_score(y_te, y_pred),
        'ROC-AUC'    : roc_auc_score(y_te, y_prob),
        'CV F1 Media': cv_scores.mean(),
        'CV F1 Std'  : cv_scores.std(),
        'modelo_obj' : modelo
    }

print('✅ Función de evaluación lista')

✅ Función de evaluación lista


In [ ]:
# COMO SE LLAMA
# lr_ridge = LogisticRegression(penalty='l2', C=0.1, class_weight='balanced', max_iter=1000)

# # Esto lo uso para llamar a la función de calculo
# res_lr = evaluar_modelo('Logistic_Regression', lr_ridge,
#                         X_train_scaled, X_test_scaled, y_train, y_test)

# print('📊 Logistic Regression:')
# # Esto imprime las variables de return
# for k, v in res_lr.items():
#     if k not in ['Modelo', 'modelo_obj']:
#         print(f'  {k}: {v:.4f}')

#### Evaluar modelos

In [158]:
# Evaluar Arbol de decision
clf = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
res_arbol = evaluar_modelo('Arbol de Decisión', clf, X_train, X_test, y_train, y_test)

print('📊 Arbol de Decisión:')
# Esto imprime las variables de return
for k, v in res_arbol.items():
    if k not in ['Modelo', 'modelo_obj']:
        print(f'  {k}: {v:.4f}')

📊 Arbol de Decisión:
  Accuracy: 0.9013
  Precision: 0.8634
  Recall: 0.9693
  F1: 0.9133
  ROC-AUC: 0.9471
  CV F1 Media: 0.9084
  CV F1 Std: 0.0159


In [ ]:
# Evaluar XGBoost (Ensemble)

# import xgboost as xgb

# #clf = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
# model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1)
# res_arbol = evaluar_modelo('XGBoost', model, X_train, X_test, y_train, y_test)

# print('📊 XGBoost:')
# # Esto imprime las variables de return
# for k, v in res_arbol.items():
#     if k not in ['Modelo', 'modelo_obj']:
#         print(f'  {k}: {v:.4f}')

#### Guardar el modelo entrenado